# Daily Closeout

Cierre ejecutivo de `daily` sobre el run full `v030`, separando `schema`, residuo `vw` explicable y n?cleo duro real.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
from IPython.display import clear_output, display

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 220)

ROOT = Path(r'C:/TSIS_Data/02_backtest_SmallCaps')
RUN_DIR = ROOT / 'runs/backtest/daily_v2_validation/daily_validate_2005_2026_d_full_v030'
CURRENT_PATH = RUN_DIR / 'daily_current.parquet'
LT1B_PATH = ROOT / 'runs/backtest/market_cap_last_observed_cutoff/20260320_market_cap_last_observed_cutoff/market_cap_cutoff_lt_1b_active_inactive.parquet'
VALIDATION_MANIFEST = json.loads((RUN_DIR / 'validation_run_manifest.json').read_text(encoding='utf-8'))
MATERIALIZATION_SUMMARY = json.loads((RUN_DIR / 'materialization_summary.json').read_text(encoding='utf-8'))
LT1B_TICKERS = set(pd.read_parquet(LT1B_PATH, columns=['ticker'])['ticker'].astype(str).str.upper().unique().tolist())

def load_current() -> pd.DataFrame:
    df = pd.read_parquet(CURRENT_PATH)
    df['ticker'] = df['ticker'].astype(str).str.upper()
    df = df[df['ticker'].isin(LT1B_TICKERS)].copy()
    metrics = pd.json_normalize(df['metrics_json'].map(json.loads))
    out = pd.concat([df.drop(columns=['metrics_json']).reset_index(drop=True), metrics.reset_index(drop=True)], axis=1)
    out['issues_str'] = out['issues'].astype(str)
    out['warns_str'] = out['warns'].astype(str)
    out['has_vw_issue'] = out['issues_str'].str.contains('vw_outside_range_severe', na=False)
    out['has_vw_warn'] = out['warns_str'].str.contains('vw_outside_range_', na=False)
    out['has_invalid_price_issue'] = out['issues_str'].str.contains('negative_or_zero_ohlc_rows', na=False)
    out['has_parse_issue'] = out['issues_str'].str.contains('all_rows_invalid_after_parse', na=False)
    out['vw_ratio_pct'] = np.where(out['rows_after_parse'] > 0, out['vw_outside_range_rows'] / out['rows_after_parse'] * 100.0, np.nan)
    out['n_per_day'] = np.where(out['rows_after_parse'] > 0, out['n_sum'] / out['rows_after_parse'], np.nan)
    out['v_per_day'] = np.where(out['rows_after_parse'] > 0, out['v_sum'] / out['rows_after_parse'], np.nan)
    out['daily_refined_bucket'] = np.select([
        out['has_parse_issue'] | out['has_invalid_price_issue'],
        out['has_vw_issue'] & (out['vw_ratio_pct'] < 1.0) & (out['vw_problem_days'] < 3),
        out['has_vw_issue'] & (out['vw_ratio_pct'] < 5.0) & (out['vw_problem_days'] < 10),
        out['has_vw_issue'] & (out['vw_ratio_pct'] >= 5.0) & (out['vw_ratio_pct'] < 20.0),
        out['has_vw_issue'] & (out['vw_ratio_pct'] >= 20.0),
        out['has_vw_warn'],
    ], [
        'hard_invalid_parse_or_price',
        'vw_edge_absmax_only',
        'vw_low_ratio_limited_days',
        'vw_mid_ratio_illiquid_regime',
        'vw_high_ratio_illiquid_regime',
        'vw_warn_minor_or_material',
    ], default='schema_only_or_other')
    out['quality_policy'] = np.select([
        out['daily_refined_bucket'].isin(['schema_only_or_other', 'vw_edge_absmax_only']),
        out['daily_refined_bucket'].isin(['vw_low_ratio_limited_days', 'vw_mid_ratio_illiquid_regime', 'vw_high_ratio_illiquid_regime', 'vw_warn_minor_or_material']),
        out['daily_refined_bucket'].eq('hard_invalid_parse_or_price'),
    ], ['good', 'review', 'bad'], default='review')
    return out

cur = load_current()


## Preflight


In [ ]:
selected_files_full = int(sum(batch.get('files_selected', batch.get('rows', 0)) for batch in VALIDATION_MANIFEST.get('batch_files', [])))
preflight = pd.DataFrame([{
    'current_path': str(CURRENT_PATH),
    'scope': '<1B filtered from full v030',
    'selected_files_full': selected_files_full,
    'current_rows_full': MATERIALIZATION_SUMMARY['current_rows'],
    'rows_lt1b': int(len(cur)),
    'tickers_lt1b': int(cur['ticker'].nunique()),
    'validator_version': VALIDATION_MANIFEST['validator_version'],
    'workers': VALIDATION_MANIFEST['workers'],
    'finished_at_utc': MATERIALIZATION_SUMMARY['materialized_at_utc'],
}])
display(preflight)


## Snapshot


In [ ]:
snapshot = pd.DataFrame([{
    'files_total': int(len(cur)),
    'tickers_total': int(cur['ticker'].nunique()),
    'hard_fail_files': int((cur['severity'] == 'HARD_FAIL').sum()),
    'soft_fail_files': int((cur['severity'] == 'SOFT_FAIL').sum()),
    'vw_issue_files': int(cur['has_vw_issue'].sum()),
    'invalid_parse_or_price_files': int((cur['has_parse_issue'] | cur['has_invalid_price_issue']).sum()),
}])
display(snapshot)

severity = cur['severity'].value_counts().rename_axis('severity').reset_index(name='files')
fig, ax = plt.subplots(figsize=(7,4))
sns.barplot(data=severity, x='severity', y='files', ax=ax)
ax.set_title('Daily Full v030 Severity Counts')
plt.tight_layout()


## Buckets Refinados


In [ ]:
bucket_summary = cur.groupby('daily_refined_bucket').agg(
    files=('ticker', 'size'),
    tickers=('ticker', 'nunique'),
    median_ratio_pct=('vw_ratio_pct', 'median'),
    median_problem_days=('vw_problem_days', 'median'),
    median_pct_of_vw=('vw_outside_range_pct_of_vw_median', 'median'),
    median_n_per_day=('n_per_day', 'median'),
    median_v_per_day=('v_per_day', 'median'),
).reset_index().sort_values('files', ascending=False)
bucket_summary['pct_of_total'] = bucket_summary['files'] / len(cur) * 100.0
display(bucket_summary)

fig, ax = plt.subplots(figsize=(12,4))
sns.barplot(data=bucket_summary, x='daily_refined_bucket', y='files', ax=ax, order=bucket_summary['daily_refined_bucket'])
ax.set_title('Buckets Refinados Daily')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()


## Lectura Ejecutiva


In [ ]:
executive = pd.DataFrame([
    {'claim': 'schema es conocido y transversal', 'value': int(cur['warns_str'].str.contains('dataset_read_incompatible_schema', na=False).sum())},
    {'claim': 'severe actual por borde abs_max', 'value': int(((cur['daily_refined_bucket'] == 'vw_edge_absmax_only')).sum())},
    {'claim': 'severe persistente iliquido', 'value': int((cur['daily_refined_bucket'] == 'vw_high_ratio_illiquid_regime').sum())},
    {'claim': 'hard invalid parse/precio', 'value': int((cur['daily_refined_bucket'] == 'hard_invalid_parse_or_price').sum())},
])
display(executive)


## Pol?tica Good Review Bad


In [ ]:
policy = pd.DataFrame([
    {'bucket': 'schema_only_or_other', 'policy': 'good', 'reason': 'schema known, no economic issue'},
    {'bucket': 'vw_edge_absmax_only', 'policy': 'good', 'reason': '1-2 days, edge abs gap, not persistent'},
    {'bucket': 'vw_warn_minor_or_material', 'policy': 'review', 'reason': 'vw mismatch present but below hard bucket'},
    {'bucket': 'vw_low_ratio_limited_days', 'policy': 'review', 'reason': 'few affected days, limited ratio'},
    {'bucket': 'vw_mid_ratio_illiquid_regime', 'policy': 'review', 'reason': 'persistent but typically illiquid flat-bar regime'},
    {'bucket': 'vw_high_ratio_illiquid_regime', 'policy': 'review', 'reason': 'stronger illiquid regime, keep out of core until use-case review'},
    {'bucket': 'hard_invalid_parse_or_price', 'policy': 'bad', 'reason': 'true hard invalid parse or non-positive OHLC'},
])
display(policy)


## Casos Duros Reales


In [ ]:
hard_invalid = cur[cur['daily_refined_bucket'] == 'hard_invalid_parse_or_price'].copy()
cols = ['ticker','year','file','issues','rows_after_parse','negative_or_zero_ohlc_rows','o_min','l_min','h_max','c_max']
display(hard_invalid[cols].head(40))


## Inspector Final


In [ ]:
inspect_df = cur[cur['daily_refined_bucket'].isin(['vw_edge_absmax_only','vw_low_ratio_limited_days','vw_mid_ratio_illiquid_regime','vw_high_ratio_illiquid_regime','hard_invalid_parse_or_price'])].copy()
inspect_df = inspect_df.sort_values(['daily_refined_bucket','vw_ratio_pct','vw_outside_range_abs_max'], ascending=[True, False, False])
bucket_dd = widgets.Dropdown(options=sorted(inspect_df['daily_refined_bucket'].dropna().unique().tolist()), description='bucket', layout=widgets.Layout(width='320px'))
file_dd = widgets.Dropdown(description='file', layout=widgets.Layout(width='900px'))
out = widgets.Output()

def refresh_files(*_):
    sub = inspect_df[inspect_df['daily_refined_bucket'] == bucket_dd.value].copy()
    opts = []
    for idx, row in sub.head(250).iterrows():
        label = f"{row['ticker']} | {int(row['year'])} | ratio={row.get('vw_ratio_pct', np.nan):.2f}% | abs_max={row.get('vw_outside_range_abs_max', np.nan):.4f}"
        opts.append((label, idx))
    file_dd.options = opts
    if opts: file_dd.value = opts[0][1]

def render(*_):
    with out:
        clear_output(wait=True)
        if file_dd.value is None:
            print('sin casos')
            return
        row = inspect_df.loc[file_dd.value]
        display(row[['ticker','year','daily_refined_bucket','quality_policy','issues','warns','rows_after_parse','vw_outside_range_rows','vw_problem_days','vw_ratio_pct','vw_outside_range_abs_max','vw_outside_range_pct_of_vw_max','n_per_day','v_per_day','coverage_ratio_vs_business_days','file']].to_frame().T)

bucket_dd.observe(refresh_files, names='value')
file_dd.observe(render, names='value')
refresh_files()
render()
display(widgets.VBox([widgets.HBox([bucket_dd]), file_dd, out]))
